# 4th down: compute
Works through the data and runs all simulations, then saves everything to `results/results.json`.
The plots and the handbook are in **`results.ipynb`**, which only reads `results/results.json`.

`SPEED` scales the number of simulated games: 1.0 ≈ 30 min (final numbers), 0.2 ≈ 6 min (quick test).

In [1]:
import random
import numpy as np
import pandas as pd
import model

model.SPEED = 1.0
results = {"speed": model.SPEED}

## 1. Data and probabilities
nflverse play-by-play, regular season 2018–2025. Probabilities use only neutral situations (quarters 1–3).

In [2]:
pbp = model.load_data()
params = model.estimate(pbp)

print(f"{len(pbp):,} plays, {pbp.neutral.sum():,} in neutral situations")
print(f"Field goals: {params['n_field_goals']:,} kicks, logistic b0 = {params['fg_logit'][0]:.2f}, b1 = {params['fg_logit'][1]:.3f}")
print(f"Mean punt {np.mean([kick_distance for kick_distance, return_yards in params['punts']]):.1f} yd, P(extra point) = {params['p_extra_point']:.3f}")
print(f"Drives per team per game: {params['drives_mean']:.2f} -> the simulator uses {params['drives_per_team']}")
print(f"Coaches go for it on {params['coach_go_rate']:.1%} of {params['n_fourth_downs']:,} 4th downs")
params["p_first_down"].T.round(3)

371,798 plays, 232,585 in neutral situations
Field goals: 8,241 kicks, logistic b0 = 6.18, b1 = -0.105
Mean punt 46.2 yd, P(extra point) = 0.944
Drives per team per game: 11.15 -> the simulator uses 11
Coaches go for it on 13.9% of 18,921 4th downs


ydstogo,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
mean,0.7,0.576,0.529,0.488,0.434,0.429,0.384,0.355,0.305,0.294,0.273,0.234,0.223,0.182,0.141
count,5641.0,3456.000,3178.000,3034.000,3092.000,2891.000,2731.000,2459.000,2079.000,2588.000,1132.000,931.000,754.000,625.000,638.000


## 2. Check the simulator
Coach vs coach should win 50%; simulated P(1st down) and how drives end should match the data.

In [3]:
coach_results, coach_go_share = model.run(params, model.coach, n_games=50_000, seed=1)
win_prob, lower, upper, p_value = model.coach_vs_coach_test(coach_results)
print(f"Coach vs coach: p̂ = {win_prob:.4f} [{lower:.4f}, {upper:.4f}], p-value = {p_value:.2f}")
print(f"Simulated coaches go for it on {coach_go_share:.1%} of 4th downs (data {params['coach_go_rate']:.1%})")

rng = random.Random(7)

def simulated_conversion(yards_to_go):
    gains, p_turnover = params["play_outcomes"]["equal"][(3, yards_to_go)]
    return np.mean([rng.random() >= p_turnover and gains[int(rng.random() * len(gains))] >= yards_to_go
                    for _ in range(20_000)])
conversion = params["p_first_down"].loc[1:10].copy()
conversion["simulated"] = [simulated_conversion(yards_to_go) for yards_to_go in conversion.index]

simulated_drive_ends = []
for seed in range(5000):
    model.game(params, model.coach, "equal", random.Random(seed), None, simulated_drive_ends)
    
real_drives = pbp.groupby(["game_id", "fixed_drive"]).fixed_drive_result.first()
real_drives = real_drives[~real_drives.isin(["End of half", "Opp touchdown"])]

drive_ends = pd.DataFrame({"Data": real_drives.value_counts(normalize=True),
                           "Simulator": pd.Series(simulated_drive_ends).value_counts(normalize=True)}).fillna(0)
drive_ends = drive_ends.sort_values("Data", ascending=False)

results.update(coach_vs_coach=[win_prob, lower, upper, p_value], coach_go_rate_simulated=coach_go_share,
               conversion=conversion, drive_ends=drive_ends)

display(conversion.T.round(3))
drive_ends.round(3)

Coach vs coach: p̂ = 0.4989 [0.4946, 0.5032], p-value = 0.61
Simulated coaches go for it on 16.4% of 4th downs (data 13.9%)


ydstogo,1,2,3,4,5,6,7,8,9,10
mean,0.700,0.576,0.529,0.488,0.434,0.429,0.384,0.355,0.305,0.294
count,5641.000,3456.000,3178.000,3034.000,3092.000,2891.000,2731.000,2459.000,2079.000,2588.000
simulated,0.698,0.570,0.518,0.486,0.428,0.429,0.378,0.344,0.293,0.281


,Data,Simulator
Punt,0.391,0.428
Touchdown,0.245,0.244
Field goal,0.163,0.133
Turnover,0.112,0.108
Turnover on downs,0.057,0.045
Missed field goal,0.029,0.025
Safety,0.003,0.010
Punt return TD,0.000,0.002
Muffed punt,0.000,0.005


## 3. Step 1: each strategy on its own
### 3.1 How long a field goal should our team try?
Kick a field goal from up to K yards (otherwise punt), going for it on 4th & 4 or less. The best K is used from now on.

In [4]:
fg_sweep = {}
for kick_distance in [45, 50, 53, 55, 57, 59, 61, 63, 65, 68]:
    policy = model.make_policy([model.short_yardage(4)], fg_range=kick_distance - model.FG_EXTRA_YARDS)
    fg_sweep[kick_distance] = model.mean_ci(model.run(params, policy, n_games=20_000, seed=10)[0])
    
fg_range_sweep = pd.DataFrame(fg_sweep, index=["win_prob", "ci_low", "ci_high"]).T
best_kick_distance = int(fg_range_sweep.win_prob.idxmax())
fg_range = best_kick_distance - model.FG_EXTRA_YARDS      # our field-goal range in yards to the end zone

print(f"Best K = {best_kick_distance} yards (pos ≤ {fg_range}), P(made) there = {params['p_field_goal'][fg_range]:.0%}")
results.update(fg_range_sweep=fg_range_sweep, best_kick_distance=best_kick_distance, fg_range=fg_range)

Best K = 57 yards (pos ≤ 39), P(made) there = 55%


### 3.2 Aggressiveness: go at random (strategy 4) vs go on short yardage (strategy 5)

In [5]:
def sweep(policies, seed, n_games=10_000):
    """Win probability and go-for-it share of each policy; `policies` maps a parameter value to a policy."""
    rows = []
    for parameter, policy in policies.items():
        win_results, go_share = model.run(params, policy, n_games=n_games, seed=seed)
        rows.append((parameter, go_share, *model.mean_ci(win_results)))
    return pd.DataFrame(rows, columns=["parameter", "go_share", "win_prob", "ci_low", "ci_high"])

random_go_sweep = sweep({q: model.random_go(q, fg_range) for q in np.linspace(0, 1, 11)}, seed=11)
short_yardage_sweep = sweep({x: model.make_policy([model.short_yardage(x)], fg_range) for x in [0, 1, 2, 3, 4, 5, 6, 8, 10, 15]},
                            seed=11, n_games=40_000)

coach_results, coach_go_share = model.run(params, model.coach, n_games=40_000, seed=11)

best_max_yards = int(short_yardage_sweep.loc[short_yardage_sweep.win_prob.idxmax(), "parameter"])
print(f"Best X = {best_max_yards}")

results.update(random_go_sweep=random_go_sweep, short_yardage_sweep=short_yardage_sweep,
               coach_point=[coach_go_share, coach_results.mean()], best_max_yards=best_max_yards)
short_yardage_sweep.round(3)

Best X = 5


,parameter,go_share,win_prob,ci_low,ci_high
0,0,0.000,0.464,0.459,0.468
1,1,0.140,0.503,0.498,0.507
2,2,0.232,0.519,0.514,0.523
3,3,0.323,0.527,0.522,0.531
4,4,0.408,0.531,0.526,0.535
5,5,0.492,0.533,0.528,0.538
6,6,0.568,0.532,0.527,0.537
7,8,0.706,0.519,0.514,0.524
8,10,0.821,0.504,0.499,0.509
9,15,0.927,0.478,0.474,0.483


### 3.3 Strategy 7: yards to go ≤ X and pos ≤ Y
X = 1–8 yards, Y = 10–100 in 10-yard steps (Y = 100 is "go if ≤ X" anywhere), 100,000 paired games per cell.
Boxed in the plot: cells not significantly worse than the best cell (95% CI of the paired difference reaches 0).
The best 5 are carried forward.

In [6]:
short_close_heatmap, short_close_tied = model.short_close_grid(
    params, fg_range, list(range(1, 9)), list(range(10, 101, 10)), n_games=100_000, seed=12)

top_short_close_cells = [[int(x), int(y)] for x, y in short_close_heatmap.stack().sort_values(ascending=False).index[:5]]

print("Top 5 (X, Y):", top_short_close_cells)
print(f"{len(short_close_tied)} cells not significantly worse than the best:", short_close_tied)

results.update(short_close_heatmap=short_close_heatmap, short_close_tied=short_close_tied,
               top_short_close_cells=top_short_close_cells)
short_close_heatmap.round(3)

Top 5 (X, Y): [[6, 70], [7, 70], [6, 80], [5, 70], [5, 80]]
1 cells not significantly worse than the best: [[6, 70]]


,10,20,30,40,50,60,70,80,90,100
1,0.470,0.473,0.477,0.481,0.488,0.491,0.497,0.498,0.499,0.499
2,0.473,0.477,0.482,0.490,0.500,0.504,0.512,0.512,0.513,0.513
3,0.475,0.481,0.487,0.496,0.508,0.513,0.523,0.523,0.524,0.524
4,0.477,0.483,0.490,0.501,0.515,0.521,0.531,0.530,0.529,0.529
5,0.478,0.485,0.492,0.504,0.520,0.526,0.532,0.531,0.529,0.529
6,0.479,0.486,0.495,0.507,0.523,0.529,0.535,0.532,0.529,0.529
7,0.479,0.487,0.496,0.509,0.524,0.530,0.533,0.529,0.525,0.523
8,0.479,0.487,0.496,0.508,0.525,0.529,0.529,0.525,0.520,0.518


### 3.4 Strategy 10: trailing with ≤ N drives left

In [7]:
trailing_sweep = pd.Series({max_drives: model.run(params, model.make_policy([model.trailing_late(max_drives)], fg_range),
                                                  n_games=10_000, seed=13)[0].mean()
                            for max_drives in [1, 2, 3, 4, 6, 8, 12]}, name="p̂")
best_trailing_drives = int(trailing_sweep.idxmax())
results.update(trailing_sweep=trailing_sweep, best_trailing_drives=best_trailing_drives)
trailing_sweep.to_frame().T.round(3)

,1,2,3,4,6,8,12
p̂,0.481,0.486,0.491,0.499,0.497,0.504,0.503


### 3.5 Where is no man's land? (strategy 8)
On top of "go if ≤ X yards", also go whenever A ≤ pos ≤ B and there are ≤ 10 yards to go.
In the combination search the cap is either 6 or 10 yards.
The best 5 ranges are carried forward.

In [8]:
NEAR_VALUES, FAR_VALUES = [25, 30, 35, 40, 45, 50], [35, 40, 45, 50, 55, 60, 65]
DEAD_ZONE_MAX_YARDS = 10                            # no man's land only with ≤ 10 yards to go
dead_zone_heatmap = pd.DataFrame(index=NEAR_VALUES, columns=FAR_VALUES, dtype=float)
short_yardage_only = model.make_policy([model.short_yardage(best_max_yards)], fg_range)
dead_zone_baseline = model.run(params, short_yardage_only, n_games=10_000, seed=14)[0].mean()
for near in NEAR_VALUES:
    for far in FAR_VALUES:
        if near < far:
            policy = model.make_policy([model.short_yardage(best_max_yards),
                                        model.dead_zone(near, far, DEAD_ZONE_MAX_YARDS)], fg_range)
            dead_zone_heatmap.loc[near, far] = model.run(params, policy, n_games=10_000, seed=14)[0].mean()
top_dead_zones = [[int(near), int(far)] for near, far in dead_zone_heatmap.stack().sort_values(ascending=False).index[:5]]
# Team choice: 25–55 (+0.61 pp) is replaced by the next range, 40–55 (+0.57 pp); the two are within noise.
top_dead_zones = [zone for zone in top_dead_zones if zone != [25, 55]] + [[40, 55]]
best_near, best_far = top_dead_zones[0]
print(f"Best no man's land: pos {top_dead_zones[0]} ({100 * (dead_zone_heatmap.loc[best_near, best_far] - dead_zone_baseline):+.1f} pp). Top 5: {top_dead_zones}")
results.update(dead_zone_heatmap=dead_zone_heatmap, dead_zone_baseline=dead_zone_baseline, top_dead_zones=top_dead_zones)

Best no man's land: pos [35, 55] (+0.9 pp). Top 5: [[35, 55], [35, 50], [40, 50], [30, 55], [40, 55]]


### 3.6 All single strategies (fresh random numbers, 30,000 paired games)
Rule 13 (new): when leading with ≤ 2 drives left, never go for it in our own half (pos ≥ 50).

In [9]:
(short_close_x, short_close_y), (dead_zone_near, dead_zone_far) = top_short_close_cells[0], top_dead_zones[0]
short_yardage_rule = model.short_yardage(best_max_yards)
singles = {
    "1. Real coaches": model.coach,
    "2. Never": model.make_policy([], fg_range),
    "3. Always": model.always,
    "4. Fixed rate (q = 0.2)": model.random_go(0.2, fg_range),
    f"5. Short yardage (X = {best_max_yards})": model.make_policy([short_yardage_rule], fg_range),
    "6. Past midfield": model.make_policy([model.past_midfield()], fg_range),
    f"7. Short and close (X, Y = {short_close_x}, {short_close_y})": model.make_policy([model.short_and_close(short_close_x, short_close_y)], fg_range),
    f"8. No man's land ({dead_zone_near}–{dead_zone_far}, ≤{DEAD_ZONE_MAX_YARDS} yd)": model.make_policy([model.dead_zone(dead_zone_near, dead_zone_far, DEAD_ZONE_MAX_YARDS)], fg_range),
    "9. Goal line": model.make_policy([model.goal_line()], fg_range),
    f"10. Trailing late (N = {best_trailing_drives})": model.make_policy([model.trailing_late(best_trailing_drives)], fg_range),
    "11. Score-aware FG, last 3 drives": model.make_policy([], fg_range, score_aware_fg=True),
    f"12. Score-changing FG on top of rule 5 (X = {best_max_yards})": model.make_policy([short_yardage_rule], fg_range, score_changing_fg=3),
    f"13. Leading late: no go in own half on top of rule 5 (X = {best_max_yards})": model.make_policy([short_yardage_rule], fg_range, leading_late=(2, 50)),
}
singles_table = model.compare(params, singles, seed=100)
results["singles"] = singles_table
singles_table.round(3)

,p̂,diff (pp),diff lower,diff upper,go %
strategy,,,,,
1. Real coaches,0.502,0.000,0.000,0.000,16.475
2. Never,0.461,-4.117,-4.855,-3.378,0.000
3. Always,0.447,-5.482,-6.243,-4.720,100.000
4. Fixed rate (q = 0.2),0.457,-4.537,-5.294,-3.780,19.918
5. Short yardage (X = 5),0.534,3.145,2.390,3.900,49.395
6. Past midfield,0.508,0.548,-0.206,1.302,46.108
"7. Short and close (X, Y = 6, 70)",0.538,3.555,2.800,4.310,49.420
"8. No man's land (35–55, ≤10 yd)",0.489,-1.307,-2.054,-0.560,21.535
9. Goal line,0.472,-2.973,-3.712,-2.235,2.964


## 4. Step 2: the best combination (successive halving)
Every candidate is a set of rules and their parameters. Play all on a few games, keep the best quarter,
play the survivors on 4× more (fresh) games, repeat.
Rule 13 (leading late, no go in own half) is either off or ≤ 2 drives at pos ≥ 50.
When no rule says go, every candidate kicks (field goal in range, else punt).

In [10]:
all_candidates = model.candidates(top_short_close_cells, top_dead_zones)
print(f"{len(all_candidates)} candidates")
history, game_counts, last_round = model.successive_halving(params, all_candidates, fg_range, n_games=1_000, seed=700)
effects, effects_r2 = model.rule_effects(all_candidates, history[0])
finalists = sorted(history[-1], key=history[-1].get, reverse=True)[:5]
results.update(halving_diffs=[sorted(diffs.values()) for diffs in history], halving_game_counts=game_counts,
               effects=effects, effects_r2=effects_r2)
pd.DataFrame({"candidate": [model.label(all_candidates[i]) for i in finalists],
              "last-round diff (pp)": [history[-1][i] for i in finalists]}).round(2)

4224 candidates
Round 1: 1910 distinct candidates on 1,000 games each
Round 2: 477 distinct candidates on 4,000 games each
Round 3: 119 distinct candidates on 16,000 games each
Round 4: 29 distinct candidates on 64,000 games each


,candidate,last-round diff (pp)
0,≤5 yd & pos ≤80 + no man's land 35–50 (≤6 yd) ...,4.22
1,≤5 yd & pos ≤80 + no man's land 35–50 (≤6 yd) ...,4.19
2,≤6 yd & pos ≤70 + no man's land 35–50 (≤10 yd)...,4.17
3,≤5 yd & pos ≤80 + no man's land 30–55 (≤6 yd) ...,4.15
4,≤5 yd & pos ≤80 + no man's land 35–50 (≤10 yd)...,4.14


### 4.1 Final check on fresh random numbers (50,000 paired games)
The 5 best of the last round (64,000 games) are re-tested on new games: the best of thousands of noisy candidates is usually
a bit lucky (winner's curse), a fresh test removes that bias. The best one here is our strategy.

In [11]:
single_name = f"Best single rule: 5 (X = {best_max_yards})"
finalists = sorted(history[-1], key=history[-1].get, reverse=True)[:5]
final_policies = {"1. Real coaches": model.coach, single_name: model.make_policy([short_yardage_rule], fg_range)}
for i in finalists:
    final_policies[model.label(all_candidates[i])] = model.policy_from_candidate(all_candidates[i], fg_range)
final_table = model.compare(params, final_policies, n_games=50_000, seed=900)
best_name = final_table.drop(["1. Real coaches", single_name]).sort_values("p̂").index[-1]
best = all_candidates[next(i for i in finalists if model.label(all_candidates[i]) == best_name)]
paired_diff = (model.run(params, final_policies[best_name], n_games=50_000, seed=900)[0]
               - model.run(params, final_policies[single_name], n_games=50_000, seed=900)[0])
best_vs_single = list(model.mean_ci(100 * paired_diff))
print(f"Best combination: {best_name}")
print("Best combination − best single rule = {:.2f} pp [{:.2f}, {:.2f}]".format(*best_vs_single))
results.update(final=final_table, best_name=best_name, best=best, best_vs_single=best_vs_single)
final_table.sort_values("p̂", ascending=False).round(3)

Best combination: ≤5 yd & pos ≤80 + no man's land 35–50 (≤6 yd) + trailing (≤2 drives) + score-changing FG (≤2 drives) + leading late: no go at pos ≥50 (≤2 drives) | else kick
Best combination − best single rule = 1.31 pp [1.01, 1.61]


,p̂,diff (pp),diff lower,diff upper,go %
strategy,,,,,
≤5 yd & pos ≤80 + no man's land 35–50 (≤6 yd) + trailing (≤2 drives) + score-changing FG (≤2 drives) + leading late: no go at pos ≥50 (≤2 drives) | else kick,0.542,4.008,3.428,4.588,50.300
≤5 yd & pos ≤80 + no man's land 30–55 (≤6 yd) + trailing (≤2 drives) + score-changing FG (≤3 drives) + leading late: no go at pos ≥50 (≤2 drives) | else kick,0.542,4.003,3.422,4.584,50.968
≤5 yd & pos ≤80 + no man's land 35–50 (≤6 yd) + trailing (≤2 drives) + score-changing FG (≤1 drives) + leading late: no go at pos ≥50 (≤2 drives) | else kick,0.542,3.995,3.415,4.575,50.646
≤6 yd & pos ≤70 + no man's land 35–50 (≤10 yd) + trailing (≤2 drives) + score-changing FG (≤2 drives) + leading late: no go at pos ≥50 (≤2 drives) | else kick,0.541,3.856,3.273,4.439,55.922
≤5 yd & pos ≤80 + no man's land 35–50 (≤10 yd) + trailing (≤2 drives) + score-changing FG (≤1 drives) + leading late: no go at pos ≥50 (≤2 drives) | else kick,0.540,3.768,3.186,4.350,54.785
Best single rule: 5 (X = 5),0.529,2.701,2.122,3.280,49.286
1. Real coaches,0.502,0.000,0.000,0.000,16.452


## 5. Step 3: opponent strength
Strong / weak opponent = plays only from the top / bottom third of teams by EPA per play.

In [12]:
OPPONENTS = ["weak", "equal", "strong"]
MAX_YARDS_CURVE = [0, 1, 2, 3, 4, 6, 8, 10, 15]             # X = 0 is "never"
N_GAMES_CURVE = 40_000                                      # 95% CI about ±0.65 pp per point
curve_names = [str(x) for x in MAX_YARDS_CURVE] + ["always"]
gain_by_opponent, gain_ci_low, gain_ci_high = (pd.DataFrame(index=curve_names, columns=OPPONENTS, dtype=float) for _ in range(3))
coach_win_rate = {}                                         # coaches' own win rate vs each opponent
for k, opponent in enumerate(OPPONENTS):
    coach_results = model.run(params, model.coach, opponent, n_games=N_GAMES_CURVE, seed=1000 + k)[0]
    coach_win_rate[opponent] = coach_results.mean()
    curve_policies = {str(x): model.make_policy([model.short_yardage(x)], fg_range) for x in MAX_YARDS_CURVE}
    curve_policies["always"] = model.always
    for name, policy in curve_policies.items():
        strategy_results = model.run(params, policy, opponent, n_games=N_GAMES_CURVE, seed=1000 + k)[0]
        (gain_by_opponent.loc[name, opponent], gain_ci_low.loc[name, opponent],
         gain_ci_high.loc[name, opponent]) = model.mean_ci(100 * (strategy_results - coach_results))
best_max_yards_by_opponent = {opponent: gain_by_opponent[opponent].drop("always").idxmax() for opponent in OPPONENTS}
print("Best X per opponent:", best_max_yards_by_opponent)

best_strategy_gain = pd.DataFrame({
    opponent: model.compare(params, {f"best single (≤{best_max_yards} yd)": final_policies[single_name],
                                     "best combination": final_policies[best_name]},
                            opponent, n_games=30_000, seed=1100 + k)["diff (pp)"]
    for k, opponent in enumerate(OPPONENTS)})
results.update(gain_by_opponent=gain_by_opponent, gain_ci_low=gain_ci_low, gain_ci_high=gain_ci_high,
               coach_win_rate=coach_win_rate, best_max_yards_by_opponent=best_max_yards_by_opponent,
               best_strategy_gain_by_opponent=best_strategy_gain)
best_strategy_gain.round(1)

Best X per opponent: {'weak': '4', 'equal': '6', 'strong': '6'}


,weak,equal,strong
strategy,,,
best single (≤5 yd),1.5,3.1,3.6
best combination,2.9,4.1,4.9


## 6. Save

In [13]:
model.save_results(results, "results.json")
print("Saved results/results.json")

Saved results/results.json
